In [1]:
# Importing tools and loading our clean data from S3

import pandas as pd
import numpy as np
import boto3
import io
import warnings
warnings.filterwarnings('ignore')

# Your bucket name
BUCKET_NAME = "churn-mlops-project-cherry"

# Connect to S3
s3 = boto3.client('s3')

# Load the clean data we saved in Step 1
obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key='data/cleaned/netflix_churn_cleaned.csv'
)
df = pd.read_csv(io.BytesIO(obj['Body'].read()))

print(f"✅ Clean data loaded from S3!")
print(f"📊 Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

✅ Clean data loaded from S3!
📊 Shape: (1139, 11)

Columns: ['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days', 'region', 'monthly_fee', 'churned', 'payment_method', 'number_of_profiles', 'avg_watch_time_per_day']


In [2]:
# FEATURE ENGINEERING
# We are creating NEW columns from existing ones
# These new columns give the model extra clues about who might churn

# --- New Feature 1: engagement_score ---
# A customer who watches a lot and logs in recently is very engaged
# Low engagement = high chance of churning
df['engagement_score'] = (
    df['watch_hours'] * 0.5 +
    df['avg_watch_time_per_day'] * 30 -
    df['last_login_days'] * 0.3
)
print("✅ Created: engagement_score")

# --- New Feature 2: is_low_engagement ---
# Simple flag: 1 if customer barely watches anything
df['is_low_engagement'] = (df['avg_watch_time_per_day'] < 0.5).astype(int)
print("✅ Created: is_low_engagement")

# --- New Feature 3: is_long_inactive ---
# Simple flag: 1 if customer hasn't logged in for over 30 days
df['is_long_inactive'] = (df['last_login_days'] > 30).astype(int)
print("✅ Created: is_long_inactive")

# --- New Feature 4: fee_per_profile ---
# How much is each profile costing?
# High cost per profile = more likely to cancel
df['fee_per_profile'] = df['monthly_fee'] / df['number_of_profiles']
print("✅ Created: fee_per_profile")

# --- New Feature 5: watch_to_fee_ratio ---
# Are they getting value for money?
# Low watch hours but high fee = likely to churn
df['watch_to_fee_ratio'] = df['watch_hours'] / (df['monthly_fee'] + 1)
print("✅ Created: watch_to_fee_ratio")

# --- New Feature 6: age_group ---
# Group customers by age
# Different age groups behave differently
def get_age_group(age):
    if age < 25:
        return 'young'
    elif age < 40:
        return 'adult'
    elif age < 55:
        return 'middle_aged'
    else:
        return 'senior'

df['age_group'] = df['age'].apply(get_age_group)
print("✅ Created: age_group")

print(f"\n📊 New shape with extra features: {df.shape}")
print(f"\nAll columns now: {df.columns.tolist()}")

✅ Created: engagement_score
✅ Created: is_low_engagement
✅ Created: is_long_inactive
✅ Created: fee_per_profile
✅ Created: watch_to_fee_ratio
✅ Created: age_group

📊 New shape with extra features: (1139, 17)

All columns now: ['age', 'gender', 'subscription_type', 'watch_hours', 'last_login_days', 'region', 'monthly_fee', 'churned', 'payment_method', 'number_of_profiles', 'avg_watch_time_per_day', 'engagement_score', 'is_low_engagement', 'is_long_inactive', 'fee_per_profile', 'watch_to_fee_ratio', 'age_group']


In [3]:
# ENCODING
# ML models only understand numbers, not text
# So we convert text columns like 'gender', 'region' into numbers
# This is called One-Hot Encoding

# These are all the text columns we need to convert
categorical_columns = [
    'gender',
    'subscription_type',
    'region',
    'payment_method',
    'age_group'
]

# pd.get_dummies converts each text value into its own column
# Example: gender = Female becomes gender_Female = 1, gender_Male = 0
df_encoded = pd.get_dummies(df, columns=categorical_columns)

print("✅ Text columns converted to numbers!")
print(f"📊 Shape before encoding: {df.shape}")
print(f"📊 Shape after encoding:  {df_encoded.shape}")
print(f"\nAll columns after encoding:")
for col in df_encoded.columns:
    print(f"  - {col}")

✅ Text columns converted to numbers!
📊 Shape before encoding: (1139, 17)
📊 Shape after encoding:  (1139, 33)

All columns after encoding:
  - age
  - watch_hours
  - last_login_days
  - monthly_fee
  - churned
  - number_of_profiles
  - avg_watch_time_per_day
  - engagement_score
  - is_low_engagement
  - is_long_inactive
  - fee_per_profile
  - watch_to_fee_ratio
  - gender_Female
  - gender_Male
  - gender_Other
  - subscription_type_Basic
  - subscription_type_Premium
  - subscription_type_Standard
  - region_Africa
  - region_Asia
  - region_Europe
  - region_North America
  - region_Oceania
  - region_South America
  - payment_method_Credit Card
  - payment_method_Crypto
  - payment_method_Debit Card
  - payment_method_Gift Card
  - payment_method_PayPal
  - age_group_adult
  - age_group_middle_aged
  - age_group_senior
  - age_group_young


In [4]:
# Now we split our data into:
# X = all the input columns (what we give the model to learn from)
# y = the target column (what we want the model to predict)

# Target column - what we are predicting
TARGET = 'churned'

# Features - everything except the target
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

print(f"✅ Features and target separated!")
print(f"📊 X shape (features): {X.shape}")
print(f"📊 y shape (target):   {y.shape}")
print(f"\nFeature columns ({len(X.columns)} total):")
for col in X.columns:
    print(f"  - {col}")
print(f"\nTarget: {TARGET}")
print(f"  Stayed  (0): {(y==0).sum()}")
print(f"  Churned (1): {(y==1).sum()}")

✅ Features and target separated!
📊 X shape (features): (1139, 32)
📊 y shape (target):   (1139,)

Feature columns (32 total):
  - age
  - watch_hours
  - last_login_days
  - monthly_fee
  - number_of_profiles
  - avg_watch_time_per_day
  - engagement_score
  - is_low_engagement
  - is_long_inactive
  - fee_per_profile
  - watch_to_fee_ratio
  - gender_Female
  - gender_Male
  - gender_Other
  - subscription_type_Basic
  - subscription_type_Premium
  - subscription_type_Standard
  - region_Africa
  - region_Asia
  - region_Europe
  - region_North America
  - region_Oceania
  - region_South America
  - payment_method_Credit Card
  - payment_method_Crypto
  - payment_method_Debit Card
  - payment_method_Gift Card
  - payment_method_PayPal
  - age_group_adult
  - age_group_middle_aged
  - age_group_senior
  - age_group_young

Target: churned
  Stayed  (0): 574
  Churned (1): 565


In [5]:
# We split data into two parts:
# Training set (80%) = the model learns from this
# Test set (20%)     = we use this to check how good the model is
# 
# Think of it like studying from a textbook (train)
# and then taking an exam with new questions (test)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% goes to testing
    random_state=42,      # makes results reproducible
    stratify=y            # makes sure both sets have equal churn ratio
)

print(f"✅ Data split into train and test!")
print(f"📊 Training rows:   {len(X_train)}")
print(f"📊 Testing rows:    {len(X_test)}")
print(f"\nTraining churn rate:  {y_train.mean()*100:.1f}%")
print(f"Testing churn rate:   {y_test.mean()*100:.1f}%")
print("(These should be similar - good sign! ✅)")

✅ Data split into train and test!
📊 Training rows:   911
📊 Testing rows:    228

Training churn rate:  49.6%
Testing churn rate:   49.6%
(These should be similar - good sign! ✅)


In [6]:
# SCALING
# Some columns have very different ranges
# age is between 18-80, watch_hours could be 0-100
# Scaling makes all numbers fall in the same range
# This helps the model learn better and faster

from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

# Fit the scaler on training data and transform both train and test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to dataframe so column names are preserved
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Save the scaler - we need it later when serving predictions
joblib.dump(scaler, '/tmp/scaler.pkl')
print("✅ Data scaled!")
print("✅ Scaler saved to /tmp/scaler.pkl")
print(f"\nSample of scaled data:")
print(X_train_scaled.head(3))

✅ Data scaled!
✅ Scaler saved to /tmp/scaler.pkl

Sample of scaled data:
        age  watch_hours  last_login_days  monthly_fee  number_of_profiles  \
0 -1.725032    -0.317390         1.755857     1.112186            0.688808   
1  1.277260     0.295092        -1.497695     1.112186           -1.405199   
2  0.298252    -0.606407         1.581559    -1.363734            1.386810   

   avg_watch_time_per_day  engagement_score  is_low_engagement  \
0               -0.367309         -0.519366           0.732295   
1                1.165512          1.236213          -1.365569   
2               -0.393646         -0.557168           0.732295   

   is_long_inactive  fee_per_profile  ...  region_South America  \
0          0.987997        -0.415554  ...             -0.456314   
1         -1.012148         2.571008  ...             -0.456314   
2          0.987997        -1.013088  ...              2.191472   

   payment_method_Credit Card  payment_method_Crypto  \
0                   -0.4

In [7]:
# Save all our processed data to S3
# The training notebook will read from here

def save_to_s3(data, bucket, key):
    csv_buffer = io.StringIO()
    data.to_csv(csv_buffer, index=False)
    s3.put_object(Bucket=bucket, Key=key, Body=csv_buffer.getvalue())
    print(f"✅ Saved: s3://{bucket}/{key}")

# Save train and test sets
save_to_s3(X_train_scaled, BUCKET_NAME, 'data/features/X_train.csv')
save_to_s3(X_test_scaled,  BUCKET_NAME, 'data/features/X_test.csv')
save_to_s3(y_train.reset_index(drop=True).to_frame(), BUCKET_NAME, 'data/features/y_train.csv')
save_to_s3(y_test.reset_index(drop=True).to_frame(),  BUCKET_NAME, 'data/features/y_test.csv')

# Also save the full encoded dataframe for monitoring later
save_to_s3(df_encoded, BUCKET_NAME, 'data/features/full_features.csv')

# Upload scaler to S3
s3.upload_file('/tmp/scaler.pkl', BUCKET_NAME, 'models/scaler.pkl')
print(f"✅ Saved: s3://{BUCKET_NAME}/models/scaler.pkl")

print(f"\n🎉 Step 2 Complete! Features saved to S3!")
print(f"\nYour S3 now has:")
print(f"  📁 data/features/X_train.csv  - training features")
print(f"  📁 data/features/X_test.csv   - testing features")
print(f"  📁 data/features/y_train.csv  - training labels")
print(f"  📁 data/features/y_test.csv   - testing labels")
print(f"  📁 models/scaler.pkl          - the scaler")

✅ Saved: s3://churn-mlops-project-cherry/data/features/X_train.csv
✅ Saved: s3://churn-mlops-project-cherry/data/features/X_test.csv
✅ Saved: s3://churn-mlops-project-cherry/data/features/y_train.csv
✅ Saved: s3://churn-mlops-project-cherry/data/features/y_test.csv
✅ Saved: s3://churn-mlops-project-cherry/data/features/full_features.csv
✅ Saved: s3://churn-mlops-project-cherry/models/scaler.pkl

🎉 Step 2 Complete! Features saved to S3!

Your S3 now has:
  📁 data/features/X_train.csv  - training features
  📁 data/features/X_test.csv   - testing features
  📁 data/features/y_train.csv  - training labels
  📁 data/features/y_test.csv   - testing labels
  📁 models/scaler.pkl          - the scaler
